# Regularization

Deep Learning models have so much flexibility and capacity that **overfitting can be a serious problem**, if the training dataset is not big enough. Sure it does well on the training set, but the learned network **doesn't generalize to new examples** that it has never seen!

**You will learn to:** Use regularization in your deep learning models.

Let's get started!

## Table of Contents
- [1 - Packages](#1)
- [2 - Problem Statement](#2)
- [3 - Loading the Dataset](#3)
- [4 - Non-Regularized Model](#4)
- [5 - L2 Regularization](#5)
    - [Exercise 1 - compute_cost_with_regularization](#ex-1)
    - [Exercise 2 - backward_propagation_with_regularization](#ex-2)
- [6 - Dropout](#6)
    - [6.1 - Forward Propagation with Dropout](#6-1)
        - [Exercise 3 - forward_propagation_with_dropout](#ex-3)
    - [6.2 - Backward Propagation with Dropout](#6-2)
        - [Exercise 4 - backward_propagation_with_dropout](#ex-4)
- [7 - Conclusions](#7)

<a name='1'></a>
## 1 - Packages

In [1]:
# Colab: download the original auxiliary scripts from the public repository.
import os, sys, subprocess
from pathlib import Path
BASE = "https://raw.githubusercontent.com/agrptec/assignment4-raisin-regularization/main/scripts"
Path("scripts").mkdir(exist_ok=True)
for name in ("reg_utils.py", "testCases.py", "public_tests.py"):
    subprocess.run(["wget", "-q", "-O", f"scripts/{name}", f"{BASE}/{name}"], check=True)
sys.path.insert(0, str(Path("scripts").resolve()))
%pip -q install ucimlrepo


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import packages
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import sklearn.datasets
import scipy.io
from reg_utils import sigmoid, relu, plot_decision_boundary, initialize_parameters, load_2D_dataset, predict_dec
from reg_utils import compute_cost, predict, forward_propagation, backward_propagation, update_parameters
from testCases import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

In [ ]:
# [AGREGADO] Compatibilidad: reg_utils.predict usa np.int, que NumPy >= 1.24 eliminó.
# Sin esta línea, predict() falla con AttributeError en NumPy moderno. No cambia ningún resultado.
if not hasattr(np, 'int'):
    np.int = int


<a name='2'></a>
## 2 - Problem Statement

Raisin (UCI): classify Kecimen vs Besni using seven morphological features.


<a name='3'></a>
## 3 - Loading the Dataset

In [ ]:
# UCI Raisin dataset: binary classification; 60/20/20 stratified split.
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
raisin = fetch_ucirepo(id=850)
X = raisin.data.features.to_numpy(dtype=float)
y = raisin.data.targets.iloc[:, 0].map({"Kecimen": 0, "Besni": 1}).to_numpy(dtype=int)
assert np.isfinite(X).all() and not np.isnan(y).any()
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
Xval, Xte, yval, yte = train_test_split(Xtmp, ytmp, test_size=0.5, stratify=ytmp, random_state=42)
scaler = StandardScaler()
train_X, train_Y = scaler.fit_transform(Xtr).T, ytr.reshape(1, -1)
val_X, val_Y = scaler.transform(Xval).T, yval.reshape(1, -1)
test_X, test_Y = scaler.transform(Xte).T, yte.reshape(1, -1)
print("Train / validation / test:", train_X.shape, val_X.shape, test_X.shape)


The input has seven standardized features; the model performs binary classification.


The dataset is divided into stratified training, validation and held-out test sets.


<a name='4'></a>
## 4 - Non-Regularized Model

You will use the following neural network (already implemented for you below). This model can be used:
- in *regularization mode* -- by setting the `lambd` input to a non-zero value. We use "`lambd`" instead of "`lambda`" because "`lambda`" is a reserved keyword in Python.
- in *dropout mode* -- by setting the `keep_prob` to a value less than one

You will first try the model without any regularization. Then, you will implement:
- *L2 regularization* -- functions: "`compute_cost_with_regularization()`" and "`backward_propagation_with_regularization()`"
- *Dropout* -- functions: "`forward_propagation_with_dropout()`" and "`backward_propagation_with_dropout()`"

In each part, you will run this model with the correct inputs so that it calls the functions you've implemented. Take a look at the code below to familiarize yourself with the model.

In [ ]:
def model(X, Y, learning_rate = 0.3, num_iterations = 30000, print_cost = True, lambd = 0, keep_prob = 1):
    """
    Implements a three-layer neural network: LINEAR->RELU->LINEAR->RELU->LINEAR->SIGMOID.

    Arguments:
    X -- input data, of shape (input size, number of examples)
    Y -- true "label" vector (1 for blue dot / 0 for red dot), of shape (output size, number of examples)
    learning_rate -- learning rate of the optimization
    num_iterations -- number of iterations of the optimization loop
    print_cost -- If True, print the cost every 10000 iterations
    lambd -- regularization hyperparameter, scalar
    keep_prob - probability of keeping a neuron active during drop-out, scalar.

    Returns:
    parameters -- parameters learned by the model. They can then be used to predict.
    """

    grads = {}
    costs = []                            # to keep track of the cost
    m = X.shape[1]                        # number of examples
    layers_dims = [X.shape[0], 20, 3, 1]

    # Initialize parameters dictionary.
    parameters = initialize_parameters(layers_dims)

    # Loop (gradient descent)

    for i in range(0, num_iterations):

        # Forward propagation: LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID.
        if keep_prob == 1:
            a3, cache = forward_propagation(X, parameters)
        elif keep_prob < 1:
            a3, cache = forward_propagation_with_dropout(X, parameters, keep_prob)

        # Cost function
        if lambd == 0:
            cost = compute_cost(a3, Y)
        else:
            cost = compute_cost_with_regularization(a3, Y, parameters, lambd)

        # Backward propagation.
        assert (lambd == 0 or keep_prob == 1)   # it is possible to use both L2 regularization and dropout,
                                                # but this assignment will only explore one at a time
        if lambd == 0 and keep_prob == 1:
            grads = backward_propagation(X, Y, cache)
        elif lambd != 0:
            grads = backward_propagation_with_regularization(X, Y, cache, lambd)
        elif keep_prob < 1:
            grads = backward_propagation_with_dropout(X, Y, cache, keep_prob)

        # Update parameters.
        parameters = update_parameters(parameters, grads, learning_rate)

        # Print the loss every 10000 iterations
        if print_cost and i % 10000 == 0:
            print("Cost after iteration {}: {}".format(i, cost))
        if print_cost and i % 1000 == 0:
            costs.append(cost)

    # plot the cost
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('iterations (x1,000)')
    plt.title("Learning rate =" + str(learning_rate))
    plt.show()

    return parameters

Let's train the model without any regularization, and observe the accuracy on the train/test sets.

In [ ]:
parameters = model(train_X, train_Y)
print ("On the training set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

The baseline below is evaluated on the training and validation sets.


In [ ]:
# 2D visualization of seven-dimensional Raisin features (PCA; visualization only).
from sklearn.decomposition import PCA
z = PCA(n_components=2).fit_transform(train_X.T)
plt.scatter(z[:, 0], z[:, 1], c=train_Y.ravel(), s=14, cmap="Spectral")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Raisin — PCA projection")
plt.show()


Regularization is evaluated by comparing training and validation accuracy.


<a name='5'></a>
## 5 - L2 Regularization

The standard way to avoid overfitting is called **L2 regularization**. It consists of appropriately modifying your cost function, from:
$$J = -\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small  y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} \tag{1}$$
To:
$$J_{regularized} = \small \underbrace{-\frac{1}{m} \sum\limits_{i = 1}^{m} \large{(}\small y^{(i)}\log\left(a^{[L](i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right) \large{)} }_\text{cross-entropy cost} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2} }_\text{L2 regularization cost} \tag{2}$$

Let's modify your cost and observe the consequences.

<a name='ex-1'></a>
### Exercise 1 - compute_cost_with_regularization
Implement `compute_cost_with_regularization()` which computes the cost given by formula (2). To calculate $\sum\limits_k\sum\limits_j W_{k,j}^{[l]2}$  , use :
```python
np.sum(np.square(Wl))
```
Note that you have to do this for $W^{[1]}$, $W^{[2]}$ and $W^{[3]}$, then sum the three terms and multiply by $ \frac{1}{m} \frac{\lambda}{2} $.

In [ ]:
# GRADED FUNCTION: compute_cost_with_regularization

def compute_cost_with_regularization(A3, Y, parameters, lambd):
    """
    Implement the cost function with L2 regularization. See formula (2) above.

    Arguments:
    A3 -- post-activation, output of forward propagation, of shape (output size, number of examples)
    Y -- "true" labels vector, of shape (output size, number of examples)
    parameters -- python dictionary containing parameters of the model

    Returns:
    cost - value of the regularized loss function (formula (2))
    """
    m = Y.shape[1]
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    W3 = parameters["W3"]

    cross_entropy_cost = compute_cost(A3, Y) # This gives you the cross-entropy part of the cost

    #(≈ 1 lines of code)
    # L2_regularization_cost =
    # YOUR CODE STARTS HERE
    L2_regularization_cost = (np.sum(np.square(W1))+np.sum(np.square(W2))+np.sum(np.square(W3)))*(lambd/(2*m))

    # YOUR CODE ENDS HERE

    cost = cross_entropy_cost + L2_regularization_cost

    return cost

In [ ]:
A3, t_Y, parameters = compute_cost_with_regularization_test_case()
cost = compute_cost_with_regularization(A3, t_Y, parameters, lambd=0.1)
print("cost = " + str(cost))

compute_cost_with_regularization_test(compute_cost_with_regularization)

Of course, because you changed the cost, you have to change backward propagation as well! All the gradients have to be computed with respect to this new cost.

<a name='ex-2'></a>
### Exercise 2 - backward_propagation_with_regularization
Implement the changes needed in backward propagation to take into account regularization. The changes only concern dW1, dW2 and dW3. For each, you have to add the regularization term's gradient ($\frac{d}{dW} ( \frac{1}{2}\frac{\lambda}{m}  W^2) = \frac{\lambda}{m} W$).

In [ ]:
# GRADED FUNCTION: backward_propagation_with_regularization

def backward_propagation_with_regularization(X, Y, cache, lambd):
    """
    Implements the backward propagation of our baseline model to which we added an L2 regularization.

    Arguments:
    X -- input dataset, of shape (input size, number of examples)
    Y -- "true" labels vector, of shape (output size, number of examples)
    cache -- cache output from forward_propagation()
    lambd -- regularization hyperparameter, scalar

    Returns:
    gradients -- A dictionary with the gradients with respect to each parameter, activation and pre-activation variables
    """

    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    #(≈ 1 lines of code)
    # dW3 = 1./m * np.dot(dZ3, A2.T) + None
    # YOUR CODE STARTS HERE
    dW3 = 1./m * np.dot(dZ3, A2.T) + lambd/m*W3

    # YOUR CODE ENDS HERE
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    #(≈ 1 lines of code)
    # dW2 = 1./m * np.dot(dZ2, A1.T) + None
    # YOUR CODE STARTS HERE
    dW2 = 1./m * np.dot(dZ2, A1.T) + lambd/m*W2

    # YOUR CODE ENDS HERE
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    #(≈ 1 lines of code)
    # dW1 = 1./m * np.dot(dZ1, X.T) + None
    # YOUR CODE STARTS HERE
    dW1 = 1./m * np.dot(dZ1, X.T) + lambd/m*W1

    # YOUR CODE ENDS HERE
    db1 = 1. / m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1,
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}

    return gradients

In [ ]:
t_X, t_Y, cache = backward_propagation_with_regularization_test_case()

grads = backward_propagation_with_regularization(t_X, t_Y, cache, lambd = 0.7)
print ("dW1 = \n"+ str(grads["dW1"]))
print ("dW2 = \n"+ str(grads["dW2"]))
print ("dW3 = \n"+ str(grads["dW3"]))
backward_propagation_with_regularization_test(backward_propagation_with_regularization)

Let's now run the model with L2 regularization $(\lambda = 0.7)$. The `model()` function will call:
- `compute_cost_with_regularization` instead of `compute_cost`
- `backward_propagation_with_regularization` instead of `backward_propagation`

In [ ]:
parameters = model(train_X, train_Y, lambd = 0.7)
print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

The L2 result above is specific to the Raisin dataset and must be interpreted after execution.


In [ ]:
# 2D visualization of seven-dimensional Raisin features (PCA; visualization only).
from sklearn.decomposition import PCA
z = PCA(n_components=2).fit_transform(train_X.T)
plt.scatter(z[:, 0], z[:, 1], c=train_Y.ravel(), s=14, cmap="Spectral")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Raisin — PCA projection")
plt.show()


L2 strength is selected using validation accuracy.


<a name='6'></a>
## 6 - Dropout

Finally, **dropout** is a widely used regularization technique that is specific to deep learning.
**It randomly shuts down some neurons in each iteration.** Watch these two videos to see what this means!

<!--
To understand drop-out, consider this conversation with a friend:
- Friend: "Why do you need all these neurons to train your network and classify images?".
- You: "Because each neuron contains a weight and can learn specific features/details/shape of an image. The more neurons I have, the more featurse my model learns!"
- Friend: "I see, but are you sure that your neurons are learning different features and not all the same features?"
- You: "Good point... Neurons in the same layer actually don't talk to each other. It should be definitly possible that they learn the same image features/shapes/forms/details... which would be redundant. There should be a solution."
!-->


<center>
<video width="620" height="440" src="images/dropout1_kiank.mp4" type="video/mp4" controls>
</video>
</center>
<br>
<caption><center><font color='purple'><b>Figure 2 </b>: <b>Drop-out on the second hidden layer.</b> <br> At each iteration, you shut down (= set to zero) each neuron of a layer with probability $1 - keep\_prob$ or keep it with probability $keep\_prob$ (50% here). The dropped neurons don't contribute to the training in both the forward and backward propagations of the iteration. </font></center></caption>

<center>
<video width="620" height="440" src="images/dropout2_kiank.mp4" type="video/mp4" controls>
</video>
</center>

<caption><center><font color='purple'><b>Figure 3</b>:<b> Drop-out on the first and third hidden layers. </b><br> $1^{st}$ layer: we shut down on average 40% of the neurons.  $3^{rd}$ layer: we shut down on average 20% of the neurons. </font></center></caption>


When you shut some neurons down, you actually modify your model. The idea behind drop-out is that at each iteration, you train a different model that uses only a subset of your neurons. With dropout, your neurons thus become less sensitive to the activation of one other specific neuron, because that other neuron might be shut down at any time.

<a name='6-1'></a>
### 6.1 - Forward Propagation with Dropout

<a name='ex-3'></a>
### Exercise 3 - forward_propagation_with_dropout

Implement the forward propagation with dropout. You are using a 3 layer neural network, and will add dropout to the first and second hidden layers. We will not apply dropout to the input layer or output layer.

**Instructions**:
You would like to shut down some neurons in the first and second layers. To do that, you are going to carry out 4 Steps:
1. In lecture, we dicussed creating a variable $d^{[1]}$ with the same shape as $a^{[1]}$ using `np.random.rand()` to randomly get numbers between 0 and 1. Here, you will use a vectorized implementation, so create a random matrix $D^{[1]} = [d^{[1](1)} d^{[1](2)} ... d^{[1](m)}] $ of the same dimension as $A^{[1]}$.
2. Set each entry of $D^{[1]}$ to be 1 with probability (`keep_prob`), and 0 otherwise.

**Hint:** Let's say that keep_prob = 0.8, which means that we want to keep about 80% of the neurons and drop out about 20% of them.  We want to generate a vector that has 1's and 0's, where about 80% of them are 1 and about 20% are 0.
This python statement:  
`X = (X < keep_prob).astype(int)`  

is conceptually the same as this if-else statement (for the simple case of a one-dimensional array) :

```
for i,v in enumerate(x):
    if v < keep_prob:
        x[i] = 1
    else: # v >= keep_prob
        x[i] = 0
```
Note that the `X = (X < keep_prob).astype(int)` works with multi-dimensional arrays, and the resulting output preserves the dimensions of the input array.

Also note that without using `.astype(int)`, the result is an array of booleans `True` and `False`, which Python automatically converts to 1 and 0 if we multiply it with numbers.  (However, it's better practice to convert data into the data type that we intend, so try using `.astype(int)`.)

3. Set $A^{[1]}$ to $A^{[1]} * D^{[1]}$. (You are shutting down some neurons). You can think of $D^{[1]}$ as a mask, so that when it is multiplied with another matrix, it shuts down some of the values.
4. Divide $A^{[1]}$ by `keep_prob`. By doing this you are assuring that the result of the cost will still have the same expected value as without drop-out. (This technique is also called inverted dropout.)

In [ ]:
# GRADED FUNCTION: forward_propagation_with_dropout

def forward_propagation_with_dropout(X, parameters, keep_prob = 0.5):
    """
    Implements the forward propagation: LINEAR -> RELU + DROPOUT -> LINEAR -> RELU + DROPOUT -> LINEAR -> SIGMOID.

    Arguments:
    X -- input dataset, of shape (2, number of examples)
    parameters -- python dictionary containing your parameters "W1", "b1", "W2", "b2", "W3", "b3":
                    W1 -- weight matrix of shape (20, 2)
                    b1 -- bias vector of shape (20, 1)
                    W2 -- weight matrix of shape (3, 20)
                    b2 -- bias vector of shape (3, 1)
                    W3 -- weight matrix of shape (1, 3)
                    b3 -- bias vector of shape (1, 1)
    keep_prob - probability of keeping a neuron active during drop-out, scalar

    Returns:
    A3 -- last activation value, output of the forward propagation, of shape (1,1)
    cache -- tuple, information stored for computing the backward propagation
    """

    np.random.seed(1)

    # retrieve parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    #(≈ 4 lines of code)         # Steps 1-4 below correspond to the Steps 1-4 described above.
    # D1 =                                           # Step 1: initialize matrix D1 = np.random.rand(..., ...)
    # D1 =                                           # Step 2: convert entries of D1 to 0 or 1 (using keep_prob as the threshold)
    # A1 =                                           # Step 3: shut down some neurons of A1
    # A1 =                                           # Step 4: scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    D1 = np.random.rand(A1.shape[0],A1.shape[1])
    D1 = (D1 < keep_prob).astype(int)
    A1 = A1 * D1
    A1 = A1/keep_prob

    # YOUR CODE ENDS HERE
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    #(≈ 4 lines of code)
    # D2 =                                           # Step 1: initialize matrix D2 = np.random.rand(..., ...)
    # D2 =                                           # Step 2: convert entries of D2 to 0 or 1 (using keep_prob as the threshold)
    # A2 =                                           # Step 3: shut down some neurons of A2
    # A2 =                                           # Step 4: scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    D2 = np.random.rand(A2.shape[0],A2.shape[1])
    D2 = (D2 < keep_prob).astype(int)
    A2 = A2 * D2
    A2 = A2/keep_prob

    # YOUR CODE ENDS HERE
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    cache = (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)

    return A3, cache

In [ ]:
t_X, parameters = forward_propagation_with_dropout_test_case()

A3, cache = forward_propagation_with_dropout(t_X, parameters, keep_prob=0.7)
print ("A3 = " + str(A3))

forward_propagation_with_dropout_test(forward_propagation_with_dropout)

<a name='6-2'></a>
### 6.2 - Backward Propagation with Dropout

<a name='ex-4'></a>
### Exercise 4 - backward_propagation_with_dropout
Implement the backward propagation with dropout. As before, you are training a 3 layer network. Add dropout to the first and second hidden layers, using the masks $D^{[1]}$ and $D^{[2]}$ stored in the cache.

**Instruction**:
Backpropagation with dropout is actually quite easy. You will have to carry out 2 Steps:
1. You had previously shut down some neurons during forward propagation, by applying a mask $D^{[1]}$ to `A1`. In backpropagation, you will have to shut down the same neurons, by reapplying the same mask $D^{[1]}$ to `dA1`.
2. During forward propagation, you had divided `A1` by `keep_prob`. In backpropagation, you'll therefore have to divide `dA1` by `keep_prob` again (the calculus interpretation is that if $A^{[1]}$ is scaled by `keep_prob`, then its derivative $dA^{[1]}$ is also scaled by the same `keep_prob`).


In [ ]:
# GRADED FUNCTION: backward_propagation_with_dropout

def backward_propagation_with_dropout(X, Y, cache, keep_prob):
    """
    Implements the backward propagation of our baseline model to which we added dropout.

    Arguments:
    X -- input dataset, of shape (2, number of examples)
    Y -- "true" labels vector, of shape (output size, number of examples)
    cache -- cache output from forward_propagation_with_dropout()
    keep_prob - probability of keeping a neuron active during drop-out, scalar

    Returns:
    gradients -- A dictionary with the gradients with respect to each parameter, activation and pre-activation variables
    """

    m = X.shape[1]
    (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    dW3 = 1./m * np.dot(dZ3, A2.T)
    db3 = 1./m * np.sum(dZ3, axis=1, keepdims=True)
    dA2 = np.dot(W3.T, dZ3)
    #(≈ 2 lines of code)
    # dA2 =                # Step 1: Apply mask D2 to shut down the same neurons as during the forward propagation
    # dA2 =                # Step 2: Scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    dA2 = D2 * dA2
    dA2 = dA2 / keep_prob
    # YOUR CODE ENDS HERE
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1./m * np.dot(dZ2, A1.T)
    db2 = 1./m * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    #(≈ 2 lines of code)
    # dA1 =                # Step 1: Apply mask D1 to shut down the same neurons as during the forward propagation
    # dA1 =                # Step 2: Scale the value of neurons that haven't been shut down
    # YOUR CODE STARTS HERE
    dA1 = D1 * dA1
    dA1 = dA1 /keep_prob

    # YOUR CODE ENDS HERE
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1./m * np.dot(dZ1, X.T)
    db1 = 1./m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1,
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}

    return gradients

In [ ]:
t_X, t_Y, cache = backward_propagation_with_dropout_test_case()

gradients = backward_propagation_with_dropout(t_X, t_Y, cache, keep_prob=0.8)

print ("dA1 = \n" + str(gradients["dA1"]))
print ("dA2 = \n" + str(gradients["dA2"]))

backward_propagation_with_dropout_test(backward_propagation_with_dropout)

Let's now run the model with dropout (`keep_prob = 0.86`). It means at every iteration you shut down each neurons of layer 1 and 2 with 14% probability. The function `model()` will now call:
- `forward_propagation_with_dropout` instead of `forward_propagation`.
- `backward_propagation_with_dropout` instead of `backward_propagation`.

In [ ]:
parameters = model(train_X, train_Y, keep_prob = 0.86, learning_rate = 0.3)

print ("On the train set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

The dropout result above is specific to the Raisin dataset and must be interpreted after execution.


In [ ]:
# 2D visualization of seven-dimensional Raisin features (PCA; visualization only).
from sklearn.decomposition import PCA
z = PCA(n_components=2).fit_transform(train_X.T)
plt.scatter(z[:, 0], z[:, 1], c=train_Y.ravel(), s=14, cmap="Spectral")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Raisin — PCA projection")
plt.show()


**Note**:
- A **common mistake** when using dropout is to use it both in training and testing. You should use dropout (randomly eliminate nodes) only in training.
- Deep learning frameworks like [tensorflow](https://www.tensorflow.org/api_docs/python/tf/nn/dropout), [PaddlePaddle](http://doc.paddlepaddle.org/release_doc/0.9.0/doc/ui/api/trainer_config_helpers/attrs.html), [keras](https://keras.io/layers/core/#dropout) or [caffe](http://caffe.berkeleyvision.org/tutorial/layers/dropout.html) come with a dropout layer implementation. Don't stress - you will soon learn some of these frameworks.

<font color='blue'>
    
**What you should remember about dropout:**
- Dropout is a regularization technique.
- You only use dropout during training. Don't use dropout (randomly eliminate nodes) during test time.
- Apply dropout both during forward and backward propagation.
- During training time, divide each dropout layer by keep_prob to keep the same expected value for the activations. For example, if keep_prob is 0.5, then we will on average shut down half the nodes, so the output will be scaled by 0.5 since only the remaining half are contributing to the solution. Dividing by 0.5 is equivalent to multiplying by 2. Hence, the output now has the same expected value. You can check that this works even when keep_prob is other values than 0.5.  

<a name='7'></a>
## 7 - Conclusions

The comparison is performed on the Raisin dataset; consult the experiment tables below.


Regularization can reduce overfitting, but it can also increase bias.


End of the original tutorial section; the experiments below compare additional techniques.


The experiments below are retained from the team notebook and adapted to Raisin.


---
# Parte 2 — Extensión: L1, barridos de hiperparámetros y aumento de datos

Las secciones 1–7 de arriba son el tutorial original (baseline, L2 y dropout con un solo valor de hiperparámetro). Aquí se prueban **las cuatro técnicas pedidas** con varios valores de hiperparámetro.

**Protocolo (igual para todas las técnicas)**
- Dataset 2D del notebook: 211 puntos de entrenamiento y 200 de **validación** (el "test set" del notebook se usa aquí como validación).
- Red `2 → 20 → 3 → 1` (ReLU, ReLU, sigmoide), `learning_rate = 0.3`, 30 000 iteraciones.
- Cada configuración se entrena con **5 inicializaciones distintas** (`SEEDS`; la `3` es la del notebook original) y se reporta **media ± desviación estándar**. Con solo 200 puntos de validación, una corrida individual (1 punto = 0.5 %) mide más ruido que efecto.
- Las precisiones de entrenamiento se miden **siempre sobre los 211 puntos originales**, también con aumento de datos, para que sean comparables.
- *Brecha* = precisión de entrenamiento − precisión de validación.

**Limitación:** no hay un conjunto de prueba aparte, así que elegir el mejor hiperparámetro con la validación da una cifra algo optimista.

In [ ]:
# ============ Utilidades de la Parte 2 ============
import time
import multiprocessing as mp
import pandas as pd
from reg_utils import compute_cost, backward_propagation, forward_propagation, update_parameters, relu, sigmoid

 # val_X and val_Y were defined in the stratified split above
SEEDS = [3, 11, 23, 42, 77]              # seed=3 reproduce la inicialización del notebook original


def init_params(layers, seed):
    """Igual que reg_utils.initialize_parameters (que fija seed=3), pero con seed configurable."""
    np.random.seed(seed)
    p = {}
    for l in range(1, len(layers)):
        p["W%d" % l] = np.random.randn(layers[l], layers[l - 1]) / np.sqrt(layers[l - 1])
        p["b%d" % l] = np.zeros((layers[l], 1))
    return p


def cross_entropy(A3, Y, eps=1e-12):
    """Entropía cruzada media con clipping (solo para reportar; evita inf si la salida satura)."""
    A = np.clip(A3, eps, 1 - eps)
    return float(np.mean(-(Y * np.log(A) + (1 - Y) * np.log(1 - A))))


def accuracy(X, Y, params):
    A3, _ = forward_propagation(X, params)      # evaluación SIN dropout
    return float(np.mean((A3 > 0.5) == Y))


def train(X, Y, Xv, Yv, reg="none", lambd=0.0, keep_prob=1.0, lr=0.3, iters=30000, seed=3,
          dropout_mode="stochastic", layers=(20, 3, 1), curve_every=500, X_orig=None, Y_orig=None):
    """
    reg: 'none' | 'l2' | 'l1' | 'dropout'.
    dropout_mode: 'stochastic' (máscara nueva en cada iteración) | 'fixed' (la función del notebook, que reinicia seed(1)).
    X_orig, Y_orig: entrenamiento ORIGINAL; las métricas de entrenamiento se reportan sobre él.
    """
    X_orig = X if X_orig is None else X_orig
    Y_orig = Y if Y_orig is None else Y_orig
    params = init_params([X.shape[0], *layers], seed)
    rng = np.random.RandomState(seed + 1000)
    curve = []
    for i in range(iters):
        if reg == "dropout":
            if dropout_mode == "fixed":
                A3, cache = forward_propagation_with_dropout(X, params, keep_prob)
            else:
                A3, cache = forward_propagation_with_dropout_stochastic(X, params, keep_prob, rng)
            grads = backward_propagation_with_dropout(X, Y, cache, keep_prob)
        else:
            A3, cache = forward_propagation(X, params)
            if reg == "l2":
                grads = backward_propagation_with_regularization(X, Y, cache, lambd)
            elif reg == "l1":
                grads = backward_propagation_with_l1(X, Y, cache, lambd)
            else:
                grads = backward_propagation(X, Y, cache)
        params = update_parameters(params, grads, lr)
        if i % curve_every == 0 or i == iters - 1:
            curve.append((i, accuracy(X_orig, Y_orig, params), accuracy(Xv, Yv, params)))

    W = np.concatenate([params["W%d" % l].ravel() for l in (1, 2, 3)])
    A3t, _ = forward_propagation(X_orig, params)
    A3v, _ = forward_propagation(Xv, params)
    metrics = dict(train_acc=accuracy(X_orig, Y_orig, params), val_acc=accuracy(Xv, Yv, params),
                   train_ce=cross_entropy(A3t, Y_orig), val_ce=cross_entropy(A3v, Yv),
                   w_l1=float(np.abs(W).sum()), w_l2=float(np.sqrt((W ** 2).sum())),
                   frac_small=float(np.mean(np.abs(W) < 1e-2)), curve=curve)
    return params, metrics


def fit(cfg, seed):
    """Entrena una configuración (aplicando aumento de datos si corresponde) y devuelve (parámetros, métricas)."""
    X, Y = train_X, train_Y
    if cfg.get("aug") == "jitter":
        X, Y = augment_jitter(train_X, train_Y, cfg["k"], cfg["sigma"], seed=seed)
    elif cfg.get("aug") == "interp":
        X, Y = augment_interpolate(train_X, train_Y, cfg["k"], seed=seed)
    kw = {k: cfg[k] for k in ("reg", "lambd", "keep_prob", "dropout_mode") if k in cfg}
    with np.errstate(all="ignore"):
        params, m = train(X, Y, val_X, val_Y, seed=seed, X_orig=train_X, Y_orig=train_Y, **kw)
    m.update(label=cfg["label"], cfg=cfg, seed=seed, n_train_used=int(X.shape[1]))
    return params, m


def _job(task):
    cfg, seed = task
    return fit(cfg, seed)[1]


def run_sweep(cfgs, seeds=SEEDS, workers=2):
    """Entrena todas las configuraciones x semillas (en paralelo si el sistema permite 'fork')."""
    tasks = [(c, s) for c in cfgs for s in seeds]
    t0 = time.time()
    if workers > 1 and "fork" in mp.get_all_start_methods():
        with mp.get_context("fork").Pool(workers) as pool:
            res = pool.map(_job, tasks, chunksize=1)
    else:
        res = [_job(t) for t in tasks]
    print("%d entrenamientos en %.0f s" % (len(tasks), time.time() - t0))
    return res


def tabla(results, xkey=None):
    """Resumen por configuración: media y desviación estándar sobre las semillas."""
    rows = []
    for l in dict.fromkeys(r["label"] for r in results):
        rs = [r for r in results if r["label"] == l]
        g = lambda k: np.array([r[k] for r in rs])
        rows.append({"configuración": l, "x": rs[0]["cfg"].get(xkey) if xkey else np.nan,
                     "train_%": 100 * g("train_acc").mean(), "train_std": 100 * g("train_acc").std(),
                     "val_%": 100 * g("val_acc").mean(), "val_std": 100 * g("val_acc").std(),
                     "brecha": 100 * (g("train_acc").mean() - g("val_acc").mean()),
                     "val_CE": g("val_ce").mean(), "|W|_1": g("w_l1").mean(),
                     "pesos<0.01": g("frac_small").mean(), "n_train": rs[0]["n_train_used"]})
    return pd.DataFrame(rows)


def mostrar(df, cols=None):
    cols = cols or ["configuración", "train_%", "train_std", "val_%", "val_std", "brecha", "val_CE", "|W|_1", "pesos<0.01"]
    print(df[cols].round(3).to_string(index=False))


def plot_acc(df, base, xlabel, title, logx=True, ax=None):
    ax = ax or plt.gca()
    ax.errorbar(df.x, df["train_%"], df.train_std, marker="o", capsize=3, label="entrenamiento")
    ax.errorbar(df.x, df["val_%"], df.val_std, marker="s", capsize=3, label="validación")
    ax.axhline(base["val_%"], color="C1", ls="--", lw=1, label="validación sin regularizar")
    ax.axhline(base["train_%"], color="C0", ls="--", lw=1, label="entrenamiento sin regularizar")
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("precisión (%)"); ax.set_title(title)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)


def boundary(ax, params, X, Y, title):
    """PCA projection colored by model predictions (not a decision boundary)."""
    from sklearn.decomposition import PCA
    z = PCA(n_components=2).fit_transform(X.T)
    preds = (forward_propagation(X, params)[0] > 0.5).ravel()
    ax.scatter(z[:, 0], z[:, 1], c=preds, s=14, cmap=plt.cm.Spectral)
    ax.set_title(title, fontsize=9); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

plt.rcParams["figure.figsize"] = (11, 4)
print("Listo. Semillas:", SEEDS)


## 8 - Barrido de L2 (Ridge)

Se usan **tus** funciones `compute_cost_with_regularization` y `backward_propagation_with_regularization` (sección 5) sin cambios. Se prueban varios valores de $\lambda$; $\lambda = 0$ es el modelo sin regularizar.

In [ ]:
cfg_base = [dict(label="sin regularización", reg="none")]
cfg_l2 = [dict(label="L2 lambda=%g" % lam, reg="l2", lambd=lam) for lam in [0.01, 0.03, 0.1, 0.3, 0.7, 1, 3, 10]]

res_base = run_sweep(cfg_base)
res_l2 = run_sweep(cfg_l2)
df_base = tabla(res_base).iloc[0]
df_l2 = tabla(res_l2, "lambd")
mostrar(pd.concat([tabla(res_base), df_l2], ignore_index=True))

In [ ]:
fig, ax = plt.subplots(1, 2)
plot_acc(df_l2, df_base, r"$\lambda$ (L2)", "L2: precisión vs $\lambda$ (media ± std, 5 semillas)", ax=ax[0])
ax[1].plot(df_l2.x, df_l2["val_CE"], marker="o", label="L2")
ax[1].axhline(df_base["val_CE"], color="gray", ls="--", label="sin regularizar")
ax[1].set_xscale("log"); ax[1].set_xlabel(r"$\lambda$ (L2)"); ax[1].set_ylabel("entropía cruzada en validación")
ax[1].set_title("L2: pérdida de validación"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

Interpret the L2 sweep from the generated table; original dataset figures do not apply.


## 9 - Regularización L1 (Lasso)

El costo suma el **valor absoluto** de los pesos en lugar de su cuadrado:

$$J_{L1} = \underbrace{-\frac{1}{m}\sum_{i}\Big(y^{(i)}\log a^{[3](i)} + (1-y^{(i)})\log(1-a^{[3](i)})\Big)}_{\text{entropía cruzada}} + \frac{\lambda}{m}\Big(\sum|W^{[1]}| + \sum|W^{[2]}| + \sum|W^{[3]}|\Big)$$

Para L2 el término de regularización es $\frac{\lambda}{2m}\sum W^2$ y su gradiente $\frac{\lambda}{m}W$. Para L1 el gradiente pasa a ser $\frac{\lambda}{m}\,\mathrm{sign}(W)$ (subgradiente, porque $|W|$ no es derivable en 0): una fuerza de **magnitud constante** hacia cero sin importar el tamaño del peso. Por eso L1 lleva muchos pesos a ≈ 0 (modelo disperso), mientras que L2 los encoge proporcionalmente.

Solo cambian dos funciones respecto a L2: el costo y la retropropagación (las demás sirven igual).

In [ ]:
def compute_cost_with_l1(A3, Y, parameters, lambd):
    """Costo de entropía cruzada + (lambda/m) * suma de |W| de las 3 capas."""
    m = Y.shape[1]
    cross_entropy_cost = compute_cost(A3, Y)
    L1_regularization_cost = (lambd / m) * (np.sum(np.abs(parameters["W1"])) +
                                            np.sum(np.abs(parameters["W2"])) +
                                            np.sum(np.abs(parameters["W3"])))
    return cross_entropy_cost + L1_regularization_cost


def backward_propagation_with_l1(X, Y, cache, lambd):
    """Retropropagación normal + (lambda/m) * sign(W) en dW1, dW2, dW3."""
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache
    gradients = backward_propagation(X, Y, cache)                       # gradientes sin regularizar
    gradients["dW3"] = gradients["dW3"] + (lambd / m) * np.sign(W3)
    gradients["dW2"] = gradients["dW2"] + (lambd / m) * np.sign(W2)
    gradients["dW1"] = gradients["dW1"] + (lambd / m) * np.sign(W1)
    return gradients

**Verificación con gradient checking**: se compara el gradiente analítico de $W$ con la diferencia finita centrada $\frac{J(W+\varepsilon)-J(W-\varepsilon)}{2\varepsilon}$ en 24 pesos elegidos al azar. Se espera un error relativo muy pequeño (≈ 1e-7 o menos).

In [ ]:
def gradient_check(reg, lambd, seed=0):
    cost_fn = compute_cost_with_l1 if reg == "l1" else compute_cost_with_regularization
    grad_fn = backward_propagation_with_l1 if reg == "l1" else backward_propagation_with_regularization
    p = init_params([2, 20, 3, 1], 3)
    for _ in range(50):                                   # unos pasos para salir de la inicialización
        p = update_parameters(p, grad_fn(train_X, train_Y, forward_propagation(train_X, p)[1], lambd), 0.3)
    g = grad_fn(train_X, train_Y, forward_propagation(train_X, p)[1], lambd)
    rng, worst, eps = np.random.RandomState(seed), 0, 1e-6
    for l in (1, 2, 3):
        W = p["W%d" % l]
        for _ in range(8):
            i, j = rng.randint(W.shape[0]), rng.randint(W.shape[1])
            old = W[i, j]
            W[i, j] = old + eps; cp = cost_fn(forward_propagation(train_X, p)[0], train_Y, p, lambd)
            W[i, j] = old - eps; cm = cost_fn(forward_propagation(train_X, p)[0], train_Y, p, lambd)
            W[i, j] = old
            num, ana = (cp - cm) / (2 * eps), g["dW%d" % l][i, j]
            worst = max(worst, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
    return worst

for reg, lam in [("l2", 0.7), ("l1", 0.7), ("l1", 3.0)]:
    print("gradient check %s, lambda=%-4s -> error relativo máximo = %.2e" % (reg, lam, gradient_check(reg, lam)))

In [ ]:
cfg_l1 = [dict(label="L1 lambda=%g" % lam, reg="l1", lambd=lam) for lam in [0.01, 0.03, 0.1, 0.3, 0.7, 1, 2, 3, 5, 10]]
res_l1 = run_sweep(cfg_l1)
df_l1 = tabla(res_l1, "lambd")
mostrar(df_l1)

In [ ]:
fig, ax = plt.subplots(1, 2)
plot_acc(df_l1, df_base, r"$\lambda$ (L1)", "L1: precisión vs $\lambda$ (media ± std, 5 semillas)", ax=ax[0])
ax[0].set_ylim(88, 96)                                  # zona útil; el colapso para lambda grande queda fuera de escala
v5, v10 = df_l1.set_index("x").loc[5, "val_%"], df_l1.set_index("x").loc[10, "val_%"]
ax[0].set_title("L1: precisión vs $\\lambda$ (media ± std, 5 semillas)\n$\\lambda$=5 y $\\lambda$=10 colapsan: validación %.0f %% y %.0f %% (fuera de escala)" % (v5, v10), fontsize=9)
ax[1].plot(df_l1.x, 100 * df_l1["pesos<0.01"], marker="o", color="C3")
ax[1].set_xscale("log"); ax[1].set_xlabel(r"$\lambda$ (L1)"); ax[1].set_ylabel("% de pesos con |w| < 0.01")
ax[1].set_title("L1: dispersión de los pesos"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

Interpret the L1 sweep from the generated table; original dataset figures do not apply.


## 10 - Barrido de Dropout

Se usan **tus** funciones `forward_propagation_with_dropout` y `backward_propagation_with_dropout` (sección 6). Antes de barrer, un detalle del código provisto: `forward_propagation_with_dropout` empieza con `np.random.seed(1)`. Como `model()` la llama **en cada iteración**, genera **siempre las mismas máscaras** `D1`, `D2`: se apagan las mismas neuronas, para cada ejemplo, en todas las iteraciones. Sirve para que los tests sean reproducibles, pero no es dropout en sentido estricto (que muestrea una máscara nueva en cada paso).

In [ ]:
p0 = init_params([train_X.shape[0], 20, 3, 1], 3)
_, c1 = forward_propagation_with_dropout(train_X, p0, 0.8)
_, c2 = forward_propagation_with_dropout(train_X, p0, 0.8)
print("Función del notebook: ¿mismas máscaras en dos llamadas seguidas? D1:", np.array_equal(c1[1], c2[1]), "| D2:", np.array_equal(c1[6], c2[6]))


def forward_propagation_with_dropout_stochastic(X, parameters, keep_prob, rng):
    """Igual que forward_propagation_with_dropout, pero SIN reiniciar la semilla: máscara nueva en cada llamada."""
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    D1 = (rng.rand(*A1.shape) < keep_prob).astype(int)      # pasos 1-2: máscara de 0/1
    A1 = A1 * D1 / keep_prob                                # pasos 3-4: apagar y reescalar
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    D2 = (rng.rand(*A2.shape) < keep_prob).astype(int)
    A2 = A2 * D2 / keep_prob
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)
    return A3, (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)

In [ ]:
cfg_dp = [dict(label="dropout keep_prob=%g" % kp, reg="dropout", keep_prob=kp, dropout_mode="stochastic")
          for kp in [0.95, 0.9, 0.86, 0.8, 0.7, 0.6, 0.5]]
cfg_dpf = [dict(label="dropout (máscara fija del notebook) keep_prob=%g" % kp, reg="dropout", keep_prob=kp, dropout_mode="fixed")
           for kp in [0.9, 0.86, 0.8, 0.7, 0.5]]
res_dp = run_sweep(cfg_dp)
res_dpf = run_sweep(cfg_dpf)
df_dp = tabla(res_dp, "keep_prob")
df_dpf = tabla(res_dpf, "keep_prob")
print("--- dropout estocástico (máscara nueva en cada iteración) ---");  mostrar(df_dp)
print("\n--- dropout con la función del notebook (máscara fija, seed(1)) ---"); mostrar(df_dpf)

In [ ]:
fig, ax = plt.subplots(1, 2, sharey=True)
for a, d, t in [(ax[0], df_dp, "Dropout estocástico"), (ax[1], df_dpf, "Dropout del notebook (máscara fija)")]:
    plot_acc(d, df_base, "keep_prob", t, logx=False, ax=a)
    a.invert_xaxis()
plt.tight_layout(); plt.show()

Interpret the dropout sweep from the generated table; original dataset figures do not apply.


## 11 - Data augmentation
Gaussian jitter and within-class interpolation are applied only to training samples. Synthetic morphological features may not always be physically plausible.


In [ ]:
def augment_jitter(X, Y, k, sigma_rel, seed=0):
    """k copias de cada punto con ruido gaussiano de desviación sigma_rel * std(característica); misma etiqueta."""
    rng = np.random.RandomState(seed)
    std = X.std(axis=1, keepdims=True)
    Xs, Ys = [X], [Y]
    for _ in range(k):
        Xs.append(X + rng.randn(*X.shape) * sigma_rel * std)
        Ys.append(Y.copy())
    return np.concatenate(Xs, axis=1), np.concatenate(Ys, axis=1)


def augment_interpolate(X, Y, k, n_neighbors=5, seed=0):
    """Estilo SMOTE: k puntos nuevos por punto original, interpolando con un vecino cercano de la misma clase."""
    rng = np.random.RandomState(seed)
    Xs, Ys = [X], [Y]
    for c in np.unique(Y):
        idx = np.where(Y[0] == c)[0]
        Xc = X[:, idx]
        d = ((Xc[:, :, None] - Xc[:, None, :]) ** 2).sum(axis=0)
        np.fill_diagonal(d, np.inf)
        nn = np.argsort(d, axis=1)[:, :n_neighbors]
        for _ in range(k):
            j = nn[np.arange(len(idx)), rng.randint(0, n_neighbors, len(idx))]
            lam = rng.rand(1, len(idx))
            Xs.append(Xc * lam + Xc[:, j] * (1 - lam))
            Ys.append(np.full((1, len(idx)), c, dtype=Y.dtype))
    return np.concatenate(Xs, axis=1), np.concatenate(Ys, axis=1)


# Compare augmentation variants in a 2D PCA projection (visualization only).
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(train_X.T)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
for a, (Xa, Ya, t) in zip(ax, [(train_X, train_Y, "Original"),
    (*augment_jitter(train_X, train_Y, 3, 0.1, seed=3), "Gaussian jitter"),
    (*augment_interpolate(train_X, train_Y, 3, seed=3), "Interpolation")]):
    z = pca.transform(Xa.T)
    a.scatter(z[:, 0], z[:, 1], c=Ya.ravel(), s=8, cmap=plt.cm.Spectral)
    a.set_title(t, fontsize=9); a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout(); plt.show()


In [ ]:
cfg_aug = []
for k in (3, 10):
    for s in (0.02, 0.05, 0.1, 0.2):
        cfg_aug.append(dict(label="jitter k=%d sigma=%g" % (k, s), reg="none", aug="jitter", k=k, sigma=s))
    cfg_aug.append(dict(label="interpolación k=%d" % k, reg="none", aug="interp", k=k))
res_aug = run_sweep(cfg_aug)
df_aug = tabla(res_aug)
df_aug["x"] = [r["cfg"].get("sigma", np.nan) for r in [next(r for r in res_aug if r["label"] == l) for l in df_aug["configuración"]]]
mostrar(df_aug)

In [ ]:
fig, ax = plt.subplots(1, 2, sharey=True)
for a, k in zip(ax, (3, 10)):
    d = df_aug[df_aug["configuración"].str.startswith("jitter k=%d " % k)]
    plot_acc(d, df_base, "σ relativo del jitter", "Aumento (jitter), k=%d copias" % k, logx=True, ax=a)
plt.tight_layout(); plt.show()

Interpret the augmentation sweep from the generated table; original dataset figures do not apply.


## 12 - Comparación de las cuatro técnicas

Para cada técnica se toma la configuración con **mejor validación media** (criterio simple; ver la limitación del inicio) y se compara con el modelo sin regularizar.

In [ ]:
ALL = {r["label"]: r["cfg"] for r in res_base + res_l2 + res_l1 + res_dp + res_dpf + res_aug}
grupos = [("L2 (Ridge)", df_l2), ("L1 (Lasso)", df_l1),
          ("Dropout estocástico", df_dp), ("Dropout del notebook (máscara fija)", df_dpf),
          ("Aumento de datos", df_aug)]
filas = [("Sin regularización", df_base)]
for nombre, d in grupos:
    filas.append((nombre, d.loc[d["val_%"].idxmax()]))
resumen = pd.DataFrame([{"técnica": n, "mejor configuración": r["configuración"],
                         "train %": "%.1f ± %.1f" % (r["train_%"], r["train_std"]),
                         "validación %": "%.1f ± %.1f" % (r["val_%"], r["val_std"]),
                         "brecha": round(r["brecha"], 1), "val CE": round(r["val_CE"], 3)} for n, r in filas])
print(resumen.to_string(index=False))
BEST = {n: r["configuración"] for n, r in filas}

In [ ]:
# Proyecciones PCA y curvas de aprendizaje (semilla 3, la del notebook original)
mostrar_tecnicas = ["Sin regularización", "L2 (Ridge)", "L1 (Lasso)", "Dropout estocástico", "Aumento de datos"]
ajustes = {n: fit(ALL[BEST[n]] if BEST[n] in ALL else cfg_base[0], 3) for n in mostrar_tecnicas}

fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
for a, n in zip(ax, mostrar_tecnicas):
    p, m = ajustes[n]
    nombre = n if BEST[n].lower() == n.lower() else n + "\n" + BEST[n]
    boundary(a, p, train_X, train_Y, "%s\ntrain %.1f%% | val %.1f%%" % (nombre, 100 * m["train_acc"], 100 * m["val_acc"]))
fig.suptitle("Proyecciones PCA con la inicialización seed=3 (las tablas promedian 5 semillas, por eso las cifras difieren)", fontsize=9, y=1.03)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, sharey=True)
for n in mostrar_tecnicas:
    c = np.array(ajustes[n][1]["curve"])
    ax[0].plot(c[:, 0], 100 * c[:, 1], label=n); ax[1].plot(c[:, 0], 100 * c[:, 2], label=n)
for a, t in zip(ax, ["Precisión de entrenamiento (datos originales)", "Precisión de validación"]):
    a.set_title(t, fontsize=10); a.set_xlabel("iteración"); a.grid(alpha=0.3)
ax[0].set_ylabel("precisión (%)"); ax[1].legend(fontsize=7, loc="lower right"); ax[0].set_ylim(80, 100)
plt.tight_layout(); plt.show()

**Conclusions:** Compare the newly computed results. Do not reuse the accuracy values from the original football dataset.
